# Модель бота: монотонность и диапазоны входов
В ноутбуке 13 «что если» показало странность: при росте суммы кредита (платёж и цена товара растут
пропорционально, доход прежний) риск не растёт, а падает (8.9% -> 4.5% при x3). Пайплайн признаков проверен
и совпадает с обучением, значит дело в самой модели.

Гипотеза: модель выучила, что крупные абсолютные суммы (AMT_CREDIT, AMT_GOODS_PRICE) встречаются у более
состоятельных клиентов, и этот эффект перевешивает отношения к доходу, особенно в редких областях значений.

Проверяем два лекарства на dev (hold-out не трогаем):
1. монотонные ограничения LightGBM на признаки с понятным направлением;
2. отказ от абсолютных сумм (остаются отношения и доход).
Плюс: диапазоны входов для проверки ответов пользователя и состав неясных групп причин.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from src.bot_features import BOT_FEATURES, answers_to_features
from src.data import load_features, split_dev_holdout
from src.explain import binned_effect, group_breakdown, shap_sample
from src.features import ID_COL, TARGET_COL
from src.metrics import calibration_report
from src.models import cv_predict, make_bot_model, to_lgbm

PROCESSED = Path("../data/processed")
MODELS = Path("../models")
pd.set_option("display.width", 170)

df = load_features(PROCESSED / "train_features.parquet")
dev, _ = split_dev_holdout(df, PROCESSED / "holdout_ids.csv")  # hold-out отбрасываем
y = dev[TARGET_COL]

## 1. Варианты модели
Монотонность: +1 значит «риск не убывает с ростом значения», -1 значит «не возрастает».
Ограничения только там, где направление очевидно и подтверждено анализом (разделы 6 – 8 README).

In [ ]:
ABS_AMOUNTS = ["AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
MONOTONE = {
    "CREDIT_INCOME_RATIO": 1, "ANNUITY_INCOME_RATIO": 1,
    "BUREAU_HAS_OVERDUE_SUM": 1, "BUREAU_IS_ACTIVE_SUM": 1,
    "BUREAU_AMT_CREDIT_SUM_DEBT_SUM": 1, "BUREAU_LAST1Y_COUNT": 1,
    "DAYS_EMPLOYED": 1,      # дни отрицательные: ближе к нулю значит короче стаж, выше риск
    "AMT_INCOME_TOTAL": -1,
}
NO_ABS = [c for c in BOT_FEATURES if c not in ABS_AMOUNTS]

variants = {
    "V0: как сейчас": (BOT_FEATURES, None),
    "V1: монотонность": (BOT_FEATURES, MONOTONE),
    "V2: без абсолютных сумм": (NO_ABS, None),
    "V3: без абс. сумм + монотонность": (NO_ABS, MONOTONE),
}

In [ ]:
results = {}
for name, (cols, mono) in variants.items():
    oof, fa = cv_predict(lambda cols=cols, mono=mono: make_bot_model(cols, mono), to_lgbm(dev[cols]), y)
    results[name] = (oof, fa)
    print(f"{name}: AUC {fa.mean():.4f} ± {fa.std():.4f}")

In [ ]:
ref = results["V0: как сейчас"][1]
rows = []
for name, (oof, fa) in results.items():
    d = fa - ref
    cal = calibration_report(name, y.values, oof)
    rows.append({"вариант": name, "ROC AUC": round(fa.mean(), 4), "Gini": round(2 * fa.mean() - 1, 4),
                 "mean pred": cal["mean pred"], "ECE": cal["ECE"],
                 "diff vs V0": round(d.mean(), 4), "min": round(d.min(), 4), "max": round(d.max(), 4)})
pd.DataFrame(rows).set_index("вариант")

## 2. Поведение на «что если»
Базовый клиент тот же, что в ноутбуке 13. Для каждого сценария задано ожидаемое направление.

In [ ]:
fitted = {name: make_bot_model(cols, mono).fit(to_lgbm(dev[cols]), y) for name, (cols, mono) in variants.items()}

base = {"income": 150_000, "credit": 400_000, "annuity": 20_000, "goods_price": 400_000,
        "income_type": "Working", "education": "Secondary / secondary special", "employment_years": 5,
        "own_car": "N", "car_age": None, "own_realty": "Y", "contract_type": "Cash loans",
        "bureau_count": 2, "bureau_active": 1, "bureau_debt": 60_000, "bureau_overdue": False, "bureau_last_year": 0}

K = [0.5, 1.0, 1.5, 2.0, 3.0]
sweeps = {  # название: (список изменений ответов, ожидаемое направление риска)
    "сумма кредита x K (платёж и цена товара пропорционально)":
        ([dict(credit=400_000 * k, annuity=20_000 * k, goods_price=400_000 * k) for k in K], "up"),
    "платёж x K при той же сумме кредита": ([dict(annuity=20_000 * k) for k in K], "up"),
    "доход x K": ([dict(income=150_000 * k) for k in K], "down"),
    "стаж, лет: 0.5 / 1 / 3 / 5 / 10 / 20": ([dict(employment_years=k) for k in [0.5, 1, 3, 5, 10, 20]], "down"),
    "активных кредитов 0..5 из 5": ([dict(bureau_count=5, bureau_active=k) for k in range(6)], "up"),
    "долг в других банках: 0 / 50 тыс. / 150 тыс. / 500 тыс. / 1.5 млн":
        ([dict(bureau_count=3, bureau_active=2, bureau_debt=v) for v in [0, 50_000, 150_000, 500_000, 1_500_000]], "up"),
    "просрочки: нет / да": ([dict(bureau_overdue=False), dict(bureau_overdue=True)], "up"),
}


def probs(model, cols, changes):
    return [float(model.predict_proba(to_lgbm(answers_to_features({**base, **c}, features=cols)))[:, 1][0])
            for c in changes]


def monotone_ok(ps, direction, tol=0.0005):
    d = np.diff(ps) if direction == "up" else -np.diff(ps)
    return bool((d >= -tol).all())


table = {}
detail = {}
for sname, (changes, direction) in sweeps.items():
    for vname, (cols, _) in variants.items():
        ps = probs(fitted[vname], cols, changes)
        detail[(sname, vname)] = ps
        table.setdefault(sname, {})[vname] = "ok" if monotone_ok(ps, direction) else "НАРУШЕНИЕ"
pd.DataFrame(table).T

In [ ]:
# Числа по главному сценарию: сумма кредита
s = list(sweeps)[0]
print(s)
print(pd.DataFrame({v: [f"{p:.1%}" for p in detail[(s, v)]] for v in variants}, index=[f"x{k}" for k in K]))

## 3. Диапазоны входов для бота
Если ответ пользователя лежит за пределами того, что модель видела при обучении, её оценке верить нельзя.
Берём 0.5% и 99.5% квантили по dev; в боте это будет предупреждение или отказ от оценки.

In [ ]:
raw = dev[BOT_FEATURES].copy()
raw["EMPLOYMENT_YEARS"] = -raw["DAYS_EMPLOYED"] / 365
cols_r = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "EMPLOYMENT_YEARS", "OWN_CAR_AGE",
          "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM", "CREDIT_GOODS_RATIO",
          "BUREAU_COUNT", "BUREAU_IS_ACTIVE_SUM", "BUREAU_AMT_CREDIT_SUM_DEBT_SUM", "BUREAU_LAST1Y_COUNT"]
ranges = raw[cols_r].quantile([0.005, 0.5, 0.995, 0.999]).T.round(3)
ranges.columns = ["p0.5", "медиана", "p99.5", "p99.9"]
ranges

In [ ]:
# Где лежит сценарий «сумма x3»: в какой области распределения отношений
sc = answers_to_features({**base, "credit": 1_200_000, "annuity": 60_000, "goods_price": 1_200_000})
for c in ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO"]:
    v = float(sc[c].iloc[0])
    print(f"{c} = {v:.2f}: доля клиентов dev с меньшим значением {(raw[c] < v).mean():.1%}")

limits = {c: {"min": float(ranges.loc[c, "p0.5"]), "max": float(ranges.loc[c, "p99.5"])} for c in cols_r}
(MODELS / "bot_input_ranges.json").write_text(json.dumps(limits, ensure_ascii=False, indent=2))
print("\nСохранено: models/bot_input_ranges.json")

## 4. Что внутри неясных групп причин
«Особенности кредитной истории в других банках» в боте это один признак, `BUREAU_COUNT` (сколько всего
кредитов брал). Смотрим, в какую сторону он действует, чтобы написать точную формулировку.
Смотрим и состав «стажа и характера занятости»: он встречается в 71% отказов.

In [ ]:
Xs, ys, sv, base_value, proba = shap_sample(fitted["V0: как сейчас"], to_lgbm(dev[BOT_FEATURES]), y, n=20_000, seed=42)
print("--- BUREAU_COUNT: средний вклад SHAP по квантилям ---")
print(binned_effect(sv, Xs, "BUREAU_COUNT", q=5))
print("\n--- состав группы employment ---")
print(group_breakdown(sv, Xs, "employment", top=5))
print("\n--- NAME_INCOME_TYPE ---")
t = binned_effect(sv, Xs, "NAME_INCOME_TYPE")
print(t[t["count"] >= 200])

Что смотрим:
1. Какие варианты проходят все сценарии без «НАРУШЕНИЕ» и сколько AUC это стоит (`min` по фолдам).
   Если цена порядка 0.001 – 0.003, ею стоит заплатить за предсказуемое поведение.
2. Лечит ли проблему монотонность, отказ от абсолютных сумм или только их сочетание.
3. В каком проценте распределения лежит сценарий x3: если очень высоко, это экстраполяция, и поможет
   проверка диапазонов; если нет, дело в самой модели.
4. Какое направление у BUREAU_COUNT и как разумно назвать причину.